# 1 - Installation

In [ ]:
!pip install unsloth[colab-new]
!pip install --no-deps xformers==0.0.28.post3 
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 2 - Loading Data

In [ ]:
from datasets import load_dataset
import random
import re

print("Loading dataset...")
dataset = load_dataset("MatrixStudio/Codeforces-Python-Submissions", split="train")

shuffled_dataset = dataset.shuffle(seed=42)
subset_dataset = shuffled_dataset.select(range(100000))

print(f"Subset created. Total rows: {len(subset_dataset):,}")


def extract_hint(solution_code: str, topics: str) -> str:
    hints = []

    ds_map = {
        'deque':       'Consider using a queue-based (BFS) approach.',
        'heapq':       'Think about a priority queue or greedy strategy.',
        'Counter':     'Frequency counting might be the key insight.',
        'defaultdict': 'Grouping elements with a hash map could help.',
        'bisect':      'Binary search on the answer or a sorted structure.',
        'SortedList':  'Think about maintaining a sorted order efficiently.',
    }
    for token, hint in ds_map.items():
        if token in solution_code:
            hints.append(hint)
            break

    algo_map = {
        r'\bdp\b|\bmemo\b|\bcache\b':        'Dynamic programming could work here.',
        r'\bdfs\b|\bbfs\b':                  'Graph traversal is likely involved.',
        r'%\s*\d+|\bmod\b':                  'Pay attention to modular arithmetic.',
        r'\bprefix\b|\bcumsum\b|\baccum\b':  'Prefix sums might simplify the range queries.',
        r'\bunion\b|\bfind\b|\bdsu\b':       'Think about connected components (Union-Find).',
        r'\bbinary.search\b|lo.*mid.*hi':    'Binary search on the answer space.',
        r'\bsliding.window\b|l\s*,\s*r\s*=': 'A two-pointer or sliding window approach fits.',
    }
    for pattern, hint in algo_map.items():
        if re.search(pattern, solution_code, re.IGNORECASE):
            hints.append(hint)
            break

    if not hints and topics:
        hints.append(f'The key topics to focus on are: {topics}.')

    return ' '.join(hints[:2]) if hints else 'Try breaking the problem into smaller subproblems.'


def format_instruction(example):
    problem = example['problem-description']
    code    = example.get('submission', '')
    rating  = example.get('rating', 'Unknown')
    tags    = ', '.join(example.get('tags', []))
    hint    = extract_hint(code, tags)

    solution_prompt = (
        f"Instruction: You are an expert programmer. Solve the following Codeforces problem in Python.\n"
        f"Difficulty Rating: {rating}\n"
        f"Topics: {tags}\n\n"
        f"Problem:\n{problem}\n\n"
        f"Solution:\n{code}"
    )

    hint_prompt = (
        f"Instruction: You are a programming tutor. Give ONE short hint for this problem. "
        f"Do NOT give code or reveal the full solution. Just the key insight in 1-2 sentences.\n"
        f"Difficulty Rating: {rating}\n"
        f"Topics: {tags}\n\n"
        f"Problem:\n{problem}\n\n"
        f"Hint:\n{hint}"
    )

    # 30% hints, 70% full solutions
    text = hint_prompt if random.random() < 0.3 else solution_prompt
    return {"text": text}


train_dataset = subset_dataset.map(format_instruction)

print("Sample solution example:")
print(train_dataset[0]["text"][:400])
print("\nSample hint example:")
for i in range(len(train_dataset)):
    if "Hint:" in train_dataset[i]["text"]:
        print(train_dataset[i]["text"][:400])
        break

# 3 - Loading Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# 4 - Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 1,     
        gradient_accumulation_steps = 8,   
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit", 
        output_dir = "outputs",
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 2,
        average_tokens_across_devices = False,
    ),
)

trainer.train()

# 5 - Upload Model

In [ ]:
from huggingface_hub import login
from unsloth import FastLanguageModel

login("Your HF token :)")

model.save_pretrained_lora("outputs", tokenizer, save_method = "merged_16bit")

model.push_to_hub_merged(
    "David0dods/Qwen2.5-7B-Codeforces",
    tokenizer,
    save_method = "lora",
)
print("Done uploading!")